# Lesson 13 · Three matrices with "special powers" — special matrices: orthogonal preserves length, symmetric is a mirror, positive definite (PD) passes the energy test

**Learning goals (main tasks in bold)**
1. Understand the transpose `A.T`, the inverse `A⁻¹`, and `A @ A⁻¹ = I`
2. **Master orthogonal matrices**: `QᵀQ = I`, rotations preserve length; **implement `is_orthogonal(Q)`** (required for this lesson)
3. Meet symmetric matrices `Aᵀ=A` and what "their eigenvalues are real" means in engineering terms (the foundation of PCA)
4. **Preview (no need to finish it in this lesson)**: the DFT matrix is (up to scaling) a unitary matrix, which is behind `ifft(fft(x))==x` — pinned down in L21 / L37
5. **Optional / preview**: positive-definiteness and Cholesky — if you're short of time, mark the second half of this lesson as "just getting acquainted"

> **How to read it**: of the three special agents, **orthogonal is the main thread**; symmetric and positive definite are ability cards. Positive definite + Cholesky carries a lot of information and can be treated as a preview.

**Why it matters for Aurora**: windowing ≈ left-multiplying by a diagonal matrix; the DFT's relationship to unitary/orthogonal matrices guarantees that "transform, then inverse-transform, gets you back" (details in L21/L37).

← **Previous lesson** · [L12 · Matrix multiplication](L12_matrices.ipynb)

> Last lesson covered **matrix multiplication**: a matrix is a linear map, multiplication is composition, and a 2×2 example worked by hand.
> This lesson explores **special matrices**.

## This lesson's story: three matrices with "special powers"

Picture three kinds of secret agent:

**Orthogonal matrices**: they leave no trace — they rotate and reflect, but vector lengths stay the same. You can "undo" them directly: `Q⁻¹ = Qᵀ` (transpose = inverse). The DFT matrix is the complex-number cousin of this (a unitary matrix, once scaled by 1/√N), which is why ifft(fft(x)) = x is exact.

**Symmetric matrices**: like a mirror — fold along the main diagonal and the two halves match exactly, `Aᵀ = A`. The Gram matrix `XᵀX` and the covariance matrix are both symmetric. Their eigenvalues are always real (never complex), which makes analysis more intuitive.

**Positive definite matrices**: the energy is always positive — for any non-zero vector x, `xᵀAx > 0`. Cholesky decomposition succeeds ⟺ the matrix is positive definite; it's often used to check whether a covariance matrix is valid.

In this lesson you'll implement `is_orthogonal(Q)` to test whether any square matrix has the powers of the "orthogonal agent."

## 1. Three siblings: the identity, the transpose, the inverse

**The identity matrix I**: changes nothing, I @ x = x. It's the "1" of multiplication.

**The transpose Aᵀ**: turns rows into columns. Shape (m,n) → (n,m).
Geometric meaning: it moves the transformation over to the other side of the inner product without changing its value: `(Ax)·y = x·(Aᵀy)`.

**The inverse matrix A⁻¹**: satisfies `A @ A⁻¹ = I`; it "undoes" the transformation.
Inverting a general matrix takes O(n³) Gauss–Jordan elimination — a lot of computation.

**The elegance of orthogonal matrices**: Qᵀ = Q⁻¹. That means inverting needs no computation, just a transpose — O(1)!
Condition: the columns are pairwise orthogonal and each has length 1, i.e. `QᵀQ = I`.

Check:
```python
import numpy as np
Q = np.array([[0., -1.], [1., 0.]])  # 90° rotation
print(np.allclose(Q.T @ Q, np.eye(2)))  # → True
```

### Digging deeper: where does the identity `(Ax)·y = x·(Aᵀy)` come from?

This identity looks abstract, but it's really just the dot product taken apart and regrouped, with no new assumptions. Recall: the dot product `u · v` is "multiply matching positions, then add everything up," which in matrix form is `uᵀ v` (stand u up as a 1×n row, multiply by v as an n×1 column, and you get a number).

So:

`(Ax)·y = (Ax)ᵀ y`  (dot product = transpose-and-multiply, as we just said)

`(Ax)ᵀ = xᵀAᵀ`  (the rule "the transpose of a product = the transposes multiplied in reverse order")

Substituting:

`(Ax)·y = xᵀAᵀy = xᵀ(Aᵀy) = x · (Aᵀy)`

In other words, the whole identity is just two facts — "dot product = transpose-and-multiply" and "the transpose-of-a-product rule" — put together. Below we confirm it with concrete numbers.

### Digging deeper: why does `QᵀQ = I` guarantee that `QQᵀ = I` too?

An analogy: suppose you have a key, and turning it one full turn to the right in the lock opens the door ("cancelling from the right"). You might guess that turning it one full turn to the left must lock the door again ("cancelling from the left"). With ordinary locks that isn't necessarily so — but for **square** matrices (as many rows as columns) it's always true: as soon as cancelling works in one direction, it works in the other direction too.

In matrix language: `QᵀQ = I` says "apply transformation Q, then transformation Qᵀ, and it's as if nothing happened"; `QQᵀ = I` says the reverse, "apply Qᵀ, then Q, and it's also as if nothing happened." At first sight these are two independent facts, but for a square matrix they're two sides of the same coin.

Why, intuitively? Further down we'll show that "the columns are pairwise perpendicular and each has length 1" ⟺ `QᵀQ=I`. In n-dimensional space, n mutually perpendicular unit vectors have already "used up" all the directions — there's no room for an (n+1)-th new direction perpendicular to all of them. That means these n column vectors automatically form a complete coordinate system for the whole space (the technical term is an "orthonormal basis"); and once the columns form a complete coordinate system, one can prove that the rows must also be mutually perpendicular with length 1 — so `QQᵀ = I` follows. The full rigorous proof needs the tools of "invertibility" (determinants, existence of inverses), which are developed systematically in **L16–L18**; for now we just check in code that this fact always holds.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np

# Check (Ax)·y = x·(Aᵀy): plug in arbitrary numbers and see whether the two computations give the same number
A = np.array([[2.0, 1.0], [0.0, 3.0]])
x = np.array([1.0, -2.0])
y = np.array([3.0, 1.0])

lhs = (A @ x) @ y          # (Ax)·y
rhs = x @ (A.T @ y)        # x·(Aᵀy)
print('(Ax)·y  =', lhs)
print('x·(Aᵀy) =', rhs)
print('equal:', np.isclose(lhs, rhs))

# Check that when QᵀQ = I, QQᵀ also equals I (left inverse = right inverse, a property of square matrices)
theta = 0.7
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
print('\nQᵀQ =\n', np.round(Q.T @ Q, 10))
print('QQᵀ =\n', np.round(Q @ Q.T, 10))
print('Both directions are (approximately) the identity, so Qᵀ is both a left and a right inverse.')

## Notation first: look at the shape, then the operation

A vector is `(n,)`, a matrix is `(m, n)`, and matrix-times-vector turns `(n,)` into `(m,)`. When you meet a matrix operation, confirm the shapes first, then look at the numbers.

In [ ]:
import numpy as np
I = np.eye(3); print('identity I=\n', I)
A = np.array([[2.0,1.0],[1.0,3.0]])
print('transpose A.T=\n', A.T)
print('inverse A^-1=\n', np.linalg.inv(A))
print('A @ A^-1 ≈ I:\n', np.round(A @ np.linalg.inv(A), 6))

## Hands-on: the shape of a matrix, and its inverse

Look at the shapes and values of `A`, `A.T` and `A_inv @ A`; confirm that the transpose doesn't change the total number of entries, and that the inverse gets you back to the identity.

In [ ]:
import numpy as np

# Orthogonal matrix: Q^T @ Q = I, preserves the norm
theta = np.pi / 4
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
v = np.array([3.0, 4.0])
print('Q =\n', np.round(Q, 4))
print('Q^T @ Q =\n', np.round(Q.T @ Q, 10))  # should be I
print(f'|v| = {np.linalg.norm(v):.4f},  |Qv| = {np.linalg.norm(Q@v):.4f}  (rotation does not change length)')


## Code experiment: one matrix acting on different vectors

Loop over test vectors and watch how an ordinary matrix changes `np.linalg.norm` — and compare with the orthogonal matrix below.

### Side note: first meet "eigenvalues," then ask why a symmetric matrix's eigenvalues must be real

The code below uses `eigvalsh`, which brings in the word "eigenvalue" that we haven't covered yet, so let's explain it first rather than get hit by two new concepts at once. A matrix A acting on a vector v usually changes both v's direction and its length; but for certain special directions v, A only stretches or shrinks it, without changing its direction — `Av = λv`. That scaling factor λ is called an **eigenvalue**, and such a v is called an **eigenvector**. Eigenvalues/eigenvectors aren't developed systematically until the next lesson (L14); for now just knowing the names is enough to follow the code below.

The eigenvalues of a general matrix can be complex — a rotation matrix, for example, turns a vector through an angle rather than simply scaling it, so there's no real solution "with an unchanged direction"; you can only find one among the complex numbers. But for a **symmetric matrix** (`Aᵀ = A`) there's an algebraic theorem: the eigenvalues are always real, never complex.

The full proof needs complex conjugation and properties of the inner product, which is beyond this lesson; it's a rigorous theorem of linear algebra — here we treat it as an algebraic fact you can safely use, and leave the proof for when you work through a full linear-algebra textbook. The practical payoff: when you know A is symmetric, NumPy offers the dedicated `eigvalsh` (the h stands for Hermitian/symmetric), which is faster and more stable than the general `eig` and is guaranteed to return real values — no more annoying cases of "it's real in theory, but floating-point error returned a complex number with a tiny imaginary part."

In [ ]:
import numpy as np

# The eigenvalues of a symmetric matrix are always real
for A in [
    np.array([[2., 1.],[1., 3.]]),         # symmetric
    np.array([[4., -2.],[-2., 1.]]),       # symmetric (positive semi-definite, determinant = 0, smallest eigenvalue = 0)
]:
    vals = np.linalg.eigvalsh(A)           # eigvalsh is specifically for symmetric/Hermitian matrices
    print(f'A =\n{A}\neigenvalues = {vals}\n')
print('→ The eigenvalues of a symmetric matrix are all real; eigvalsh is faster and more stable than eig.')


## 2. Orthogonal matrices: columns pairwise perpendicular with length 1 → `Qᵀ Q = I`, and vector lengths are preserved

In [ ]:
theta = 1.0
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])  # a rotation matrix = orthogonal
x = np.array([3.0, 4.0])
print('QᵀQ =\n', np.round(Q.T @ Q, 6))
print('|x| =', np.linalg.norm(x), ' |Qx| =', np.linalg.norm(Q @ x), '(should be equal)')

## 3. ✏️ Your task: decide whether a matrix is orthogonal

**Line of reasoning**:
1. Definition of an orthogonal matrix: the columns are pairwise orthogonal and of unit length, which is equivalent to `QᵀQ = I`
2. Checking takes one line: compute `Q.T @ Q` and compare it with `np.eye(n)` using `np.allclose`
3. You must use `np.allclose`, not `==`: `cos(θ)` and `sin(θ)` carry floating-point error, and `1.0000000000000002 != 1.0`

**Reference input and output**: `Q=[[0,-1],[1,0]]` (90° rotation) → `True`; `M=[[1,2],[3,4]]` → `False`

> 💡 Need a hint? Once you've finished the exercise, you can compare with the reference implementation in the `solutions/` folder.


### Side note: what does the "tolerance" in `np.allclose` actually mean?

Floating-point arithmetic is like a ruler that can only read to 15–16 decimal places — the `1.0` you get from `cos(θ)` may actually be `0.9999999999999998`. Mathematically they're equal, but Python's `==` says "not equal." `np.allclose(a, b)` means "decide equality while allowing a tiny bit of error," and the exact criterion is:

```
|a - b| <= atol + rtol * |b|
```

The defaults are `rtol=1e-5` (relative error, tolerating differences of about one part in a hundred thousand) and `atol=1e-8` (an absolute floor on the error, so that the criterion doesn't break down when b is close to 0 and the `rtol * |b|` term also becomes 0).

For situations like this lesson's — "equal exactly in theory, with floating-point arithmetic adding a little error" — the defaults are more than enough. Some places later in this lesson use `atol=1e-12`, because they check algebraic identities that are "exactly equal in theory" (such as `ifft(fft(x)) == x`), and we want to confirm that the error has come down close to the precision limit of floating-point numbers themselves (about `1e-16`), not merely "roughly equal" — the tighter the tolerance, the finer the problems it can catch, but also the more likely it is to raise a "false alarm" from normal floating-point error. There's no universal standard for what's right; it depends on how much error you're willing to accept.

In [ ]:
import numpy as np

a, b = 1.0, 1.0 + 1e-9
print('allclose with the default tolerance:', np.allclose(a, b))              # True, the default tolerance is enough
print('allclose with a tightened tolerance:', np.allclose(a, b, atol=1e-12, rtol=0))  # False, demands more precision

Each entry of `Q.T @ Q` is a dot product between two column vectors: the diagonal entries are the inner product of a column with itself (should be 1, i.e. unit length), and the off-diagonal entries are inner products between different columns (should be 0, i.e. mutually perpendicular). If the whole matrix is approximately the identity, all the columns are pairwise orthogonal and of unit length.

### Work it through by hand: what does the definition of matrix multiplication actually compute?

The sentence above "reasons backwards from the result to the condition"; here we go the other way and reason **forwards** from the definition of matrix multiplication, laying out every intermediate step.

In the matrix product `C = Aᵀ B`, the entry in row i, column j of `C` is, by the definition from L12, "column i of A (which becomes row i of Aᵀ after transposing)" dotted with "column j of B." Call the two columns of `Q` `q₁` and `q₂` (so `Q = [q₁, q₂]`), and set `A = B = Q`:

`(QᵀQ)[i, j] = qᵢ · qⱼ`

This isn't a new rule; it's just the definition "matrix multiplication = rows dotted with columns," applied to the fact "row i of Aᵀ = column i of A."

Let's work it through concretely with the 90° rotation matrix `Q = [[0, -1], [1, 0]]`. Its two columns are `q₁ = [0, 1]` and `q₂ = [-1, 0]`:

- `(QᵀQ)[0,0] = q₁ · q₁ = 0×0 + 1×1 = 1`  (the first column dotted with itself: its length squared)
- `(QᵀQ)[0,1] = q₁ · q₂ = 0×(-1) + 1×0 = 0`  (the two columns are perpendicular)
- `(QᵀQ)[1,0] = q₂ · q₁ = (-1)×0 + 0×1 = 0`
- `(QᵀQ)[1,1] = q₂ · q₂ = (-1)×(-1) + 0×0 = 1`

Put together, that's exactly `[[1,0],[0,1]] = I`. That's the complete derivation of "columns pairwise perpendicular and of unit length" ⟹ `QᵀQ=I` — not a coincidence, but a consequence of the definition of matrix multiplication itself. The code below recomputes these four numbers and compares them with `Q.T @ Q`.

In [ ]:
import numpy as np

Q = np.array([[0.0, -1.0], [1.0, 0.0]])
q1, q2 = Q[:, 0], Q[:, 1]  # take out the two columns

manual = np.array([
    [q1 @ q1, q1 @ q2],
    [q2 @ q1, q2 @ q2],
])
print('by hand, (QᵀQ)[i,j] = qᵢ·qⱼ:\n', manual)
print('NumPy directly, Q.T @ Q:\n', Q.T @ Q)
print('agree:', np.allclose(manual, Q.T @ Q))

In [ ]:
def is_orthogonal(Q):
    # ✏️ TODO: return True/False
    raise NotImplementedError("implement is_orthogonal: return True iff Q.T @ Q ≈ I")


In [ ]:
Q = np.array([[0.0,-1.0],[1.0,0.0]])   # 90° rotation, orthogonal
M = np.array([[1.0, 2.0],[3.0, 4.0]])  # not orthogonal
try:
    print('Q orthogonal?', is_orthogonal(Q), '(should be True)')
    print('M orthogonal?', is_orthogonal(M), '(should be False)')
    assert is_orthogonal(Q), "Q (90° rotation) should be orthogonal — check that Q.T @ Q ≈ I"
    assert not is_orthogonal(M), "M is not orthogonal, so is_orthogonal should return False"
    print('\n✅ Passed: you can recognize orthogonal transformations.')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️  Not implemented yet: {e}\nPlease fill in the is_orthogonal function above.')


**🔗 Aurora connection**: because the (normalized) FFT/DCT matrices are orthogonal (unitary, in the complex case), the inverse transform = the (conjugate) transpose, so `ifft(fft(x)) == x`. That's what you'll verify in L39.

### Side note: what is a "unitary matrix," and how is it related to an orthogonal matrix?

**The complex-number version of "length stays the same"**: the core power of an orthogonal matrix is "keeping vector lengths unchanged," and it relies on `Qᵀ = Q⁻¹`. But the DFT matrix used by the FFT has **complex** entries like `e^(-2πijk/N)` (what complex numbers are and where Euler's formula comes from was covered in L05–L07). Once the entries of a matrix are complex, the ordinary transpose isn't enough; you need one extra operation: the **conjugate transpose**, written `Uᴴ` (sometimes `U*` or `U†`).

**How to compute the conjugate transpose**: first transpose the matrix (swap rows and columns — this step is the same as the ordinary transpose), then take the "complex conjugate" of each entry (the conjugate of the complex number `a+bi` is `a-bi`: the imaginary part flips sign; if the number was real to begin with, i.e. `b=0`, conjugating changes nothing). So:

- If U is a purely real matrix, `Uᴴ = Uᵀ` — the conjugate transpose reduces to the ordinary transpose; they're exactly the same.
- If U has complex entries, `Uᴴ = conj(Uᵀ)` — with the extra step "conjugate every entry."

**Why the extra step?** Because in the complex world the very definitions of "length" and "dot product" involve conjugation — the squared magnitude of a complex number z is `z · conj(z)`, not `z · z` (using `z·z` directly on complex numbers gives a "length" that is meaningless, or even complex). So the whole family of complex-world concepts — "dot product," "transpose," "length-preserving" — has to build "take the conjugate" in, for geometric intuitions like length and orthogonality to keep working.

A **unitary matrix** is the generalization of an "orthogonal matrix" to the complex world: it satisfies `Uᴴ U = I` (equivalently `U⁻¹ = Uᴴ`), and its effect is "the transformation keeps the length of complex vectors unchanged" — playing exactly the same role as an orthogonal matrix in the real world. You can remember it in one line: **an orthogonal matrix is the special case of a unitary matrix where everything is real** (then `Uᴴ` is just `Uᵀ`).

The entries of the DFT matrix are complex (of the form `e^(-iθ)`), so (once scaled by 1/√N) it's a unitary matrix rather than an orthogonal matrix in the strict sense; but it enjoys exactly the same "perks" as an orthogonal matrix — its inverse is its (conjugate) transpose, so you can invert it without any computation — and that's the algebraic reason `ifft(fft(x)) == x` holds exactly, without losing any information.

In [ ]:
import numpy as np

U = np.diag([1 + 0j, 1j])   # both diagonal entries have magnitude 1 — a simple unitary matrix
print('U =\n', U)
print('conjugate transpose Uᴴ = U.conj().T =\n', U.conj().T)
print('Uᴴ @ U =\n', np.round(U.conj().T @ U, 10), ' (should be I, meaning length is preserved)')

## 🎨 Picture: an orthogonal matrix (rotation) keeps length unchanged

In [ ]:
import numpy as np
from laviz import style, arrows2d
style()
th=0.9; R=np.array([[np.cos(th),-np.sin(th)],[np.sin(th),np.cos(th)]])
v=np.array([3.,1]); arrows2d([v, R@v], ['v','Qv (rotated, same length)'],
         title='orthogonal matrix = rotation/reflection');

## Opening act for positive definite matrices: what is `xᵀAx`, and why care about its sign?

### Starting with a story: the potential energy of a spring

Picture a spring compressed or stretched by a displacement x; the potential energy it stores is `E = (1/2) k x²` — whether the displacement is positive or negative, the potential energy is always positive (a spring always wants to spring back to where it was; it never stores energy "backwards"). That's the simplest version of "energy is non-negative," involving just one number, k.

Real vibrating systems often have several coupled springs (the many joints of a bridge pulling on each other, for example). Then the "spring constant" is no longer a number but a matrix K, the displacement is no longer a number but a vector x, and the potential-energy formula becomes:

`E = (1/2) xᵀ K x`

This is a **quadratic form** — a vector sandwiched by a matrix to give a single "energy"-like number. For a physically stable system (one that won't oscillate off to infinity on its own), this energy must be strictly positive for every non-zero displacement — which is where the "energy" in the name "positive definite matrix" comes from, and the intuition behind the course's definition "for every non-zero vector x, xᵀAx > 0."

### Shape first, then numbers

`xᵀAx` looks abstract, so let's sort out the shapes first. Let x be an n-dimensional column vector of shape `(n,)`, and A be `(n,n)`:

- `xᵀ` has shape `(1,n)` (one row, n columns)
- `xᵀ @ A`: `(1,n) @ (n,n) = (1,n)` (still one row)
- `(xᵀA) @ x`: `(1,n) @ (n,1) = (1,1)`, i.e. a **scalar** (a single number)

So the whole expression starts from "a vector," goes round a loop, and finally spits out just one number — that number is "the energy in this direction." Let's walk through it with concrete numbers.

In [ ]:
import numpy as np

A = np.array([[2.0, 0.0], [0.0, 3.0]])
x = np.array([1.0, 2.0])

step1 = x @ A           # xᵀA; NumPy 1-D arrays don't distinguish "row vectors" from "column vectors" — both have shape (n,)
step2 = step1 @ x       # (xᵀA) @ x, which ends up a scalar
print('x =', x, ' shape:', x.shape)
print('x @ A =', step1, ' shape:', step1.shape)
print('x @ A @ x =', step2, ' ← this is a scalar (the "energy" in this direction)')

# Try a few directions to see whether it's always positive
print()
for x in [np.array([1., 0.]), np.array([0., 1.]), np.array([1., -1.]), np.array([-3., 2.])]:
    energy = x @ A @ x
    print(f'x={x}  ->  xᵀAx = {energy:.2f}')

## 3. Positive definite (PD) matrices: the pass that gets you through Cholesky decomposition

**Algebraic definition**: a symmetric matrix $A$ is called **positive definite** (PD) if and only if, for every non-zero vector $x$:
$$x^\top A x > 0$$

Equivalent conditions (often used to check it):
- all eigenvalues $\lambda_i > 0$
- the Cholesky decomposition $A = L L^\top$ exists (`np.linalg.cholesky` doesn't raise an exception)

**Positive semi-definite (PSD)**: $x^\top A x \geq 0$, allowing the smallest eigenvalue to be 0; **indefinite**: has both positive and negative eigenvalues.

**Aurora connection**: the covariance matrix of PCA, $C = \frac{1}{N}X^\top X$, must be PSD — a matrix built as $A = M^\top M$ is automatically positive semi-definite.

**✏️ Mini exercise**: build `A = M.T @ M` from `M = np.random.randn(4, 3)` and check that it passes the Cholesky decomposition (it should be PD or PSD).

### Digging deeper: why is a matrix built as `MᵀM` "automatically" positive semi-definite?

It's no coincidence — it's an algebraic fact you can prove in a few lines. Let M be any matrix (not necessarily square), and let `A = MᵀM`.

**Step 1, A is always symmetric**: `Aᵀ = (MᵀM)ᵀ = Mᵀ(Mᵀ)ᵀ = MᵀM = A` (using "the transpose of a product = the transposes in reverse order" and "the transpose of a transpose is the original" — the two transpose rules). This also answers "why is XᵀX always a symmetric matrix" along the way — exactly the same reasoning, with X in place of M.

**Step 2, A is always positive semi-definite**: for any vector x, compute `xᵀAx` from the definition:

`xᵀAx = xᵀ(MᵀM)x = (xᵀMᵀ)(Mx) = (Mx)ᵀ(Mx) = |Mx|²`

The middle step just puts `xᵀMᵀ` back together as `(Mx)ᵀ` (the same transpose rule again). And `(Mx)ᵀ(Mx)` is precisely the vector `Mx` dotted with itself, i.e. the squared length `|Mx|²` of `Mx` — **a squared length can never be negative**, so `xᵀAx ≥ 0` holds for every x, which is exactly the definition of positive semi-definite.

(When is it "strictly" positive definite, i.e. `> 0` rather than just `≥ 0`? Exactly when `Mx = 0` has only the solution `x = 0` — that is, when the columns of M are linearly independent. The concept of "linear independence" is developed systematically from L14 onwards; for now just know the result.)

This proof is the fundamental reason that "a covariance matrix is automatically positive semi-definite": the covariance matrix `C = (1/N) XᵀX` is just the formula above with `M = X/√N`, so whatever the data X looks like, C is always positive semi-definite — which is also why "positive semi-definite" is a necessary condition for "a valid covariance matrix" (if a covariance matrix you actually computed isn't positive semi-definite, something has gone wrong in the numerics or in the data itself). The full application of covariance matrices and PCA comes in later lessons; here you only need to remember the algebraic fact "a Gram matrix is automatically positive semi-definite."

In [ ]:
import numpy as np

M = np.random.randn(4, 3)
A = M.T @ M
print('Is A symmetric?', np.allclose(A, A.T))
eigvals = np.linalg.eigvalsh(A)
print('eigenvalues of A:', np.round(eigvals, 4), ' ← should all be ≥ 0 (positive semi-definite)')

### What is Cholesky decomposition actually doing? Why does "the decomposition succeeds" mean positive definite?

**Understand it through the "square root" analogy**: for a number a, if a > 0 you can always find b with `a = b²`; if a < 0, there's no such b among the real numbers at all. Cholesky decomposition is the matrix version of "taking a square root": it splits A into `A = L Lᵀ`, where L is a **lower-triangular matrix** (everything above and to the right of the diagonal is 0). You can think of L as "the square root of the matrix A."

The key point: **this kind of split only exists when A is positive definite**. The reason is the same as in the proof above that `MᵀM` is positive semi-definite — if `A = LLᵀ`, then for any non-zero x:

`xᵀAx = xᵀ(LLᵀ)x = (Lᵀx)ᵀ(Lᵀx) = |Lᵀx|² ≥ 0`

The Cholesky algorithm itself requires the diagonal entries of L to be non-zero, which guarantees that `Lᵀx ≠ 0` whenever `x ≠ 0`, so this squared length is strictly greater than 0: `xᵀAx > 0` holds for every non-zero x, and A is positive definite. Conversely, when `np.linalg.cholesky` meets a case where "no such L can be found" (equivalent to having to take the square root of a negative number), it raises an error straight away — **that error is itself the evidence that "A is not positive definite"**, which is why the criterion "the decomposition succeeds ⟺ positive definite" in the code below is reliable.

### While we're at it: what is `try / except`?

If you've only learned `print`, variables and loops, the `try...except` in the code below may be new syntax. Its logic is "try the risky thing first, and if something goes wrong, catch it instead of letting the program fall over":

```python
try:
    risky_operation      # e.g. np.linalg.cholesky(A), which raises an error when A isn't positive definite
except SomeErrorType as e:
    # if that kind of error happens, jump here; the program doesn't stop
    fallback_operation
```

In this lesson's code specifically: `np.linalg.cholesky(A)` raises the "exception" (error) `LinAlgError` if A isn't positive definite. The line `except np.linalg.LinAlgError:` means "catch exactly this kind of error"; once it's caught, the program doesn't stop, but runs the code in the `except` block instead (here, falling back to the eigenvalues to decide between positive semi-definite and indefinite). Think of it as "assume first that it can be decomposed, and use the eigenvalues as a safety net for when the decomposition fails." Below, a simpler example (taking a square root) shows the same logic.

In [ ]:
import math

def safe_sqrt(a):
    try:
        return math.sqrt(a)      # this line raises ValueError when a < 0
    except ValueError as e:
        return f'⚠️ an error occurred but was caught: {e} (handle it another way, e.g. return None, instead of letting the program fall over)'

print(safe_sqrt(9))     # normal path: the code in try runs to the end
print(safe_sqrt(-4))    # triggers except; the program doesn't stop

In [ ]:
import numpy as np

# Positive-definiteness test: Cholesky decomposition succeeds ⟺ positive definite
mats = {
    'PD': np.array([[4., 2.],[2., 3.]]),
    'PSD': np.array([[1., 1.],[1., 1.]]),
    'indefinite': np.array([[1., 2.],[2., 1.]]),
}
for label, A in mats.items():
    try:
        np.linalg.cholesky(A)
        tag = '✅ positive definite'
    except np.linalg.LinAlgError:
        ev = np.linalg.eigvalsh(A)
        tag = '⬜ positive semi-definite' if ev.min() >= 0 else '❌ indefinite'
    print(f'{label:6s}  eigenvalues={np.round(np.linalg.eigvalsh(A),3)}  → {tag}')


## Parameter experiment: the rotation angle, and building a symmetric matrix

**Experiment 1**: in the section 2 cell (the one with `theta = 1.0`), change the rotation angle `theta` to `np.pi/4`, `np.pi` and `0.0`, running `is_orthogonal` each time — orthogonality doesn't depend on the angle; as long as it's a rotation matrix, the result is always `True`. Then swap in `np.diag([2, 3])` and confirm that a scaling matrix is not orthogonal.

**Experiment 2**: generate a random matrix `R = np.random.randn(3, 3)`, compute `S = R + R.T`, and check that `np.allclose(S, S.T)` is always `True`. This is the standard way to build a symmetric matrix from any matrix — and the eigenvector matrix of a symmetric matrix is an orthogonal matrix, which is where the two meet in eigendecomposition.

**Don't miss half the condition**: an orthogonal matrix needs its columns to be "pairwise perpendicular" *and* "each of length 1" — both conditions are essential. The two columns `[2,0]` and `[0,3]` of `np.diag([2, 3])` really are perpendicular (dot product 0), and many beginners stop checking at this point and think "it passes" — but their lengths are 2 and 3, not 1, so `QᵀQ` comes out as `[[4,0],[0,9]]` rather than `I`, and the orthogonality condition still fails. When you decide whether a matrix is orthogonal, "perpendicular" and "unit length" must both hold; remembering only "perpendicular" is the easiest trap to fall into.

## ✏️ Parameter experiment: verify ifft(fft(x)) == x on the spot

The (normalized) DFT matrix is unitary (U⁻¹ = Uᴴ), so the inverse transform is the conjugate transpose — which guarantees `ifft(fft(x)) == x`. No need to wait for L39: confirm it with numpy right now.

In [ ]:
import numpy as np

# An arbitrary real signal
x = np.array([1.0, 2.0, -1.0, 0.5, 3.0, -0.5, 0.0, 1.5])

# FFT → inverse FFT should restore it exactly
x_roundtrip = np.fft.ifft(np.fft.fft(x)).real
assert np.allclose(x, x_roundtrip, atol=1e-12), f'max error {abs(x - x_roundtrip).max():.2e}'
print(f'original signal: {x}')
print(f'ifft(fft):       {np.round(x_roundtrip, 12)}')
print(f'max error:       {abs(x - x_roundtrip).max():.2e}')
print('✅ ifft(fft(x)) == x — the unitary property verified on the spot (no need to wait for L39)')


## Wrap-up

You can now test any square matrix with `is_orthogonal(Q)`, and invert it by using `Q.T` directly instead of `np.linalg.inv(Q)`.
Aurora's `aurora.audio.transforms.dft()` uses a unitary matrix multiplication internally (twiddle factors dotted with the signal);
L39 relies directly on this property when it verifies `ifft(fft(x)) == x`.
Note: `dft_matrix(n)` isn't exported as a standalone function yet — the matrix form is written out inline in **L21**.
Next lesson moves on to eigendecomposition, where orthogonal matrices show up again as eigenvector matrices.


---
⬇️ **Completion check**: you've read the wrap-up; please finish the whiteboard challenge below before ticking the self-review.


### Side note: if one line of code gives the answer, why work it out by hand?

That's a good question, and it deserves a straight answer rather than being brushed off as "a teaching ritual." The point of working by hand isn't to "check whether the answer is right" (leave that to the code); it's to **force you to run every step of matrix multiplication through your head**: matrix multiplication is "which row times which column, and how it's added up," and only by writing out `Qᵀ = [[0,1],[-1,0]]` yourself and computing each entry of `Qᵀ @ Q` step by step does the definition "matrix multiplication = rows dotted with columns" really get carved into your intuition — the same kind of training as the earlier "work out `(QᵀQ)[i,j]=qᵢ·qⱼ` by hand," only this time you hold the pen.

The recommended approach: **derive it by hand first, then run the code to check**. If your derivation and the code disagree, go back and find which dot product went wrong, instead of skipping the hand-working step — that "it doesn't match, so find the mistake" process is where intuition really gets built.

## ✏️ Whiteboard challenge: special matrices by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: Q = [[0, -1], [1, 0]] (the 90° rotation matrix); compute Qᵀ @ Q by hand.
Qᵀ = [[0, 1], [-1, 0]]; is the result I?

**Question 2**: A = [[2, 1], [1, 3]]; is A symmetric? Is it positive definite?
Hint: symmetric: Aᵀ = A; positive definite: all eigenvalues > 0, or the Cholesky decomposition succeeds.

**Question 3**: D = [[3, 0], [0, 5]] (a diagonal matrix); is D orthogonal?
Hint: check whether the columns have length 1.

**Question 4**: Why is `ifft(fft(x)) == x` exact, with no information lost?
Hint: the (normalized) DFT matrix F is unitary (F⁻¹ = F^H), a member of the orthogonal family.

When you've finished, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: Q.T @ Q = I
Q = np.array([[0., -1.], [1., 0.]])
QtQ = Q.T @ Q
assert np.allclose(QtQ, np.eye(2), atol=1e-12)
print(f"Q1 ✅  Q.T @ Q = {QtQ}  (the identity, so Q is orthogonal)")

# Q2: symmetric + positive definite
A = np.array([[2., 1.], [1., 3.]])
sym = np.allclose(A, A.T, atol=1e-12)
try:
    np.linalg.cholesky(A)
    pd = True
except np.linalg.LinAlgError:
    pd = False
assert sym and pd, f"sym={sym}, pd={pd}"
eigvals = np.linalg.eigvalsh(A)
print(f"Q2 ✅  [[2,1],[1,3]] symmetric={sym}, positive definite={pd}, eigenvalues={np.round(eigvals,3)}")

# Q3: the diagonal matrix D is not orthogonal (column lengths ≠ 1)
D = np.array([[3., 0.], [0., 5.]])
try:
    is_orth = is_orthogonal(D)
    assert not is_orth, f"D should not be orthogonal, but is_orthogonal returned {is_orth}"
    print(f"Q3 ✅  diag(3,5) is not orthogonal (column lengths={np.linalg.norm(D[:,0])},{np.linalg.norm(D[:,1])}≠1)")
except (NotImplementedError, TypeError):
    print("⬜ Q3: implement is_orthogonal() first, then run the answer-check cell")

# Q4: the DFT is exactly invertible
x = np.array([1., 2., -1., 0.5])
reconstructed = np.fft.ifft(np.fft.fft(x))
assert np.allclose(reconstructed.real, x, atol=1e-12)
print(f"Q4 ✅  ifft(fft(x)) error={np.max(np.abs(reconstructed.real - x)):.2e} → exact (the DFT matrix is unitary)")
print("\n🎉 Special matrices whiteboard challenge passed! The essence of the three special matrices is now second nature.")

In [ ]:
# ✏️ Self-review for this lesson
l13_review = {
    "is_orthogonal_implemented":  None,  # is_orthogonal implemented and passes its asserts? True/False
    "orthogonal_inverse_trick":   None,  # understand what Q⁻¹=Qᵀ means? True/False
    "symmetric_pd_distinction":   None,  # can tell symmetric matrices from positive definite ones? True/False
    "fft_unitarity_understood":   None,  # understand that ifft(fft(x))=x comes from orthogonality? True/False
    "whiteboard_passed":          None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l13_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l13_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L13 complete! On to L14: eigenvalues and SVD')

---

→ **Next lesson** · [L14 · Eigenvalues and SVD](L14_eigen_svd.ipynb)

> Next lesson covers **eigenvalues and SVD**: finding the directions a matrix favors, and opening up the low-rank world with SVD.